# AnyCam — 03b · Step 5: improved baseline (bigger images + stronger augmentation)

First run (ResNet18, plain): cultivar accuracy 0.73, overfitting after epoch 7.
This notebook tries two fixes on the same official split:
- **A**: ResNet18, images enlarged 2x inside the model, stronger augmentation
- **B**: ConvNeXt-Tiny (closer to the paper's best model), same settings

Same setup as notebook 03: **Add Input → HyperLeaf2024 (trophy)**, **GPU T4 x2**, **Internet On**.
Run All (~30-40 min). Then save to GitHub as `kaggle_runs/03b_step5_improved_baseline.ipynb`.

In [ ]:
import os, subprocess, glob, json, pathlib
REPO = 'https://github.com/ShahnawazQ007/one-model-many-camera.git'
if not os.path.exists('/kaggle/working/anycam'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, '/kaggle/working/anycam'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/anycam', 'pull', '-q'], check=True)
os.chdir('/kaggle/working/anycam')
!pip -q install timm tifffile imagecodecs
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on the GPU accelerator!')
hits = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/train.csv', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(f), 'images'))]
assert hits, 'Add Input -> search hyperleaf -> the one with the trophy icon'
DATA = hits[0]; print('data:', DATA, '| images:', len(glob.glob(f'{DATA}/images/*.tiff')))

In [ ]:
# A: ResNet18, 2x larger images, stronger augmentation
!python experiments/train_baseline.py --data {DATA} --split official --upscale 2 --aug strong --epochs 50 --patience 12 --out runs/A_resnet18_up2_strong

In [ ]:
# B: ConvNeXt-Tiny, 2x larger images, stronger augmentation (smaller batch to fit GPU memory)
!python experiments/train_baseline.py --data {DATA} --split official --backbone convnext_tiny --upscale 2 --aug strong --batch 16 --lr 1e-4 --epochs 50 --patience 12 --out runs/B_convnext_tiny_up2_strong

In [ ]:
# comparison table + figures
!python scripts/compare_runs.py runs/*
!python scripts/plot_run.py runs/*
from IPython.display import Image, display
for f in sorted(glob.glob('runs/*/figures.png')):
    print(f); display(Image(f))

## Done
Wait until every cell has finished, then in the right panel **GitHub → File** type
`kaggle_runs/03b_step5_improved_baseline.ipynb` → **Save**. Tell Claude "done".